In [8]:
import os
import random
import unicodedata
import pandas as pd
from sklearn.model_selection import train_test_split


In [9]:
templates_data = pd.read_csv('../data/raw/sentence_template_starter.csv')
towns_data = pd.read_csv('../data/raw/french_town_start.csv')
generated_data = pd.read_csv('../data/generated/generated_french_town_dataset.csv')


In [10]:
templates_valid = templates_data[templates_data['label'] == 'VALID']
templates_invalid = templates_data[templates_data['label'] == 'INVALID']
both_templates = pd.concat([templates_valid, templates_invalid])
towns = towns_data['nom_ville'].tolist()

In [11]:
generated_data = []
total_data_number = 0

valid_number = random.randrange(1500, 2000)
invalid_number = random.randrange(1500, 2000)
lower_valid = int(valid_number * 0.15)
lower_invalid = int(invalid_number * 0.15)
without_accent = random.randrange(400, 600)

if generated_data:
    os.remove('../data/generated/generated_french_town_dataset.csv')


In [12]:
def generate_sentence(template, departure, arrival, should_lowercase):
    if should_lowercase:
        template = template.lower()
        departure_str = str(departure).lower()
        arrival_str = str(arrival).lower()
    else:
        departure_str = str(departure)
        arrival_str = str(arrival)

    return template.replace('{ville1}', departure_str).replace('{ville2}', arrival_str)


def generate_data_for_label(templates_df, count, lower_count, label, start_id):
    data = []
    for i in range(count):
        row = templates_df.sample(1).iloc[0]
        departure, arrival = random.sample(towns, 2)
        should_lowercase = i < lower_count
        sentence = generate_sentence(row['template'], departure, arrival, should_lowercase)

        if random.random() < 0.1:
            sentence = add_noise_to_sentence(sentence)

        if label == 'VALID':
            data.append({
                'sentence_id': start_id + i,
                'label': label,
                'sentence': sentence,
                'departure': departure,
                'arrival': arrival,
                'template_id': row['template_id']
            })
        else:
            data.append({
                'sentence_id': start_id + i,
                'label': label,
                'sentence': sentence,
                'departure': None,
                'arrival': None,
                'template_id': row['template_id']
            })
    return data


def generate_data_without_accent(templates_df, count, start_id):
    data = []
    lower_count = count // 2

    for i in range(count):
        row = templates_df.sample(1).iloc[0]
        departure, arrival = random.sample(towns, 2)
        should_lowercase = i < lower_count

        sentence = generate_sentence(row['template'], departure, arrival, should_lowercase)
        sentence_no_accent = unicodedata.normalize('NFD', sentence).encode('ascii', 'ignore').decode('utf-8')

        if random.random() < 0.1:
            sentence_no_accent = add_noise_to_sentence(sentence_no_accent)


        if row['label'] == 'VALID':
            data.append({
                'sentence_id': start_id + i,
                'label': row['label'],
                'sentence': sentence_no_accent,
                'departure': departure,
                'arrival': arrival,
                'template_id': row['template_id']
            })
        else:
            data.append({
                'sentence_id': start_id + i,
                'label': row['label'],
                'sentence': sentence_no_accent,
                'departure': None,
                'arrival': None,
                'template_id': row['template_id']
            })

    return data


def add_noise_to_sentence(sentence, noise_level=0.05):
    sentence = list(sentence)

    for i in range(len(sentence)):
        if random.random() < noise_level:
            operator = random.choice(['delete', 'swap', 'duplicate'])

            if operator == 'delete' and len(sentence) > 1:
                sentence[i] = ''
            elif operator == 'swap' and i < len(sentence) - 1:
                sentence[i], sentence[i + 1] = sentence[i + 1], sentence[i]

            elif operator == 'duplicate':
                sentence[i] = sentence[i] * 2

    noisy_sentence = ''.join(sentence)

    if random.random() < noise_level:
        noisy_sentence = ' ' + noisy_sentence
    if random.random() < noise_level:
        noisy_sentence = noisy_sentence + ' '

    return noisy_sentence

In [13]:
generated_data = []

valid_data = generate_data_for_label(
    templates_valid,
    valid_number,
    lower_valid,
    'VALID',
    start_id=1
)

invalid_data = generate_data_for_label(
    templates_invalid,
    invalid_number,
    lower_invalid,
    'INVALID',
    start_id=len(valid_data) + 1
)

accent_data = generate_data_without_accent(
    both_templates,
    without_accent,
    start_id=len(valid_data) + len(invalid_data) + 1
)

generated_data.extend(valid_data)
generated_data.extend(invalid_data)
generated_data.extend(accent_data)

In [14]:
df_generated = pd.DataFrame(generated_data).sample(frac=1)
df_generated.to_csv('data/generated/generated_french_town_dataset.csv', index=False)
df_generated.head()

OSError: Cannot save file into a non-existent directory: 'data/generated'

In [ ]:
df = pd.read_csv('../data/generated/generated_french_town_dataset.csv')

train_val, test = train_test_split(df, test_size=0.10, random_state=42, stratify=df['label'])
train, validation = train_test_split(train_val, test_size=0.1111, random_state=42, stratify=train_val['label'])

train.to_csv('data/processed/train_dataset.csv', index=False)
validation.to_csv('data/processed/validation_dataset.csv', index=False)
test.to_csv('data/processed/test_dataset.csv', index=False)